In [12]:
import torch 
from torch import nn
def corr2d(x,k):
    height,width = k.shape
    h = x.shape[0]-height+1
    w = x.shape[1]-width+1
    y = x.new_zeros(h,w)
    for i in range(h):
        for j in range(w):
            y[i,j] = (x[i:i+height,j:j+width]*k).sum()
            
    return y

def corr2d_multi_in(x, k):
    y = 0
    assert x.shape[0] == k.shape[0], "输入通道数不匹配"
    for x_channel,k_channel in zip(x,k):
        y += corr2d(x_channel,k_channel)
    return y
  
  
def corr2d_multi_in_out(x,k):
    return(torch.stack(tuple(corr2d_multi_in(x,k_out) for k_out in k)))

x = torch.randn(2,4,5)
k = torch.randn(4,2,1,2)
y = corr2d_multi_in_out(x,k)    


In [13]:
import torch
from torch import nn

torch.manual_seed(42)

# 两个输入通道
X = torch.randn(2, 4, 5)

# 四组卷积核，每组覆盖两个输入通道
K = torch.randn(4, 2, 1, 2)

# 你的实现
Y_manual = corr2d_multi_in_out(X, K)

# PyTorch 的实现
conv = nn.Conv2d(
    in_channels=2,
    out_channels=4,
    kernel_size=(1, 2),
    bias=False
)

# 必须使两种实现使用同一套权重
with torch.no_grad():
    conv.weight.copy_(K)

# Conv2d 需要 batch 维度
Y_torch = conv(X.unsqueeze(0)).squeeze(0)

print("手写输出:", Y_manual.shape)
print("PyTorch 输出:", Y_torch.shape)

print("数值是否接近:",
      torch.allclose(Y_manual, Y_torch, atol=1e-5))

print("最大绝对误差:",
      (Y_manual - Y_torch).abs().max().item())

手写输出: torch.Size([4, 4, 4])
PyTorch 输出: torch.Size([4, 4, 4])
数值是否接近: True
最大绝对误差: 4.76837158203125e-07
